# FIFA World Cup 2026 — Scoreboard Prediction

### Goal

Predict the final scoreline (home goals vs. away goals) for matches at the 2026 FIFA World Cup and using historical international results.

### Data

- **`df_results`** — international match results, 1872–present (`datasets/international_results-master/results.csv`), the historical record of goals scored by each side in each match.
- **`df_formernames`** — historical name changes for national teams (`datasets/international_results-master/former_names.csv`), used to reconcile teams that have been renamed over time so history isn't split across two identities.
- **`df_elo`** — Elo ratings for World Cup 2026 squads (`datasets/elo-ratings/elo_ratings_wc2026.csv`), a single strength number per team going into the tournament.

In [2]:
import pandas as pd
import numpy as np
import sklearn as skl

## Datasets analysis
Four raw sources are loaded, kept separate here, and will be merged later into
one model-ready table.

In [3]:
df_elo = pd.read_csv('datasets/elo-ratings/elo_ratings_wc2026.csv')
df_elo = df_elo[df_elo['year'] != 2026]

In [4]:
df_formernames = pd.read_csv('datasets/international_results-master/former_names.csv')

In [5]:
df_results = pd.read_csv('datasets/international_results-master/results.csv')
is_2026_wc = (df_results['tournament'] == 'FIFA World Cup') & (df_results['date'].str.startswith('2026'))
df_results = df_results[~is_2026_wc]

* `df_results`

International match results, 1872–present.
Columns: `date, home_team, away_team, home_score, away_score, tournament, city, country, neutral`. This is the core training table — one row per historical match.

**Filtering:** drop the rows for the **2026 FIFA World Cup** itself
(group stage through the final, `tournament == 'FIFA World Cup'` and
`date` in 2026). These matches can't be paired with a valid pre-match Elo rating (see `df_elo` below) and the tournament is otherwise out of scope for this version. All other 2026 matches (friendlies, qualifiers,
continental cups) and every earlier World Cup (1930–2022) are kept.

* `df_formernames`

Columns: `current, former, start_date, end_date`.
Maps old team names (e.g. Dahomey → Benin, West Germany → Germany) to their current name, so a team's history isn't split across two identities when merging with other sources.

* `df_elo`

Columns include: `year, snapshot_date, country, rating, rank, confederation, is_host, matches_total, wins, losses, draws, goals_for, goals_against, wc2026_exit_round`. One row per country **per year** (1901–2026, 48 countries) — a genuine time series, not a single snapshot, so it can be joined *as of* a match's date to give a leakage-free, pre-match strength feature for both teams.

**Filtering:** drop all **2026** rows. There's no pre-tournament snapshot for that year — only `2026-07-07` (mid-tournament, ratings already reflect games played) and `2026-12-31` (after the tournament is over). Neither is a valid "before kickoff" rating, so 2026 is dropped entirely; the last usable snapshot is `2025-12-31`.

### Making main data set

##### Team name reconciliation

In [6]:
name_map = dict(zip(df_formernames['former'], df_formernames['current']))

df_results['home_team'] = df_results['home_team'].replace(name_map)
df_results['away_team'] = df_results['away_team'].replace(name_map)

df_elo['country'] = df_elo['country'].replace({'Czechia': 'Czech Republic'})

##### Preparing for the join

In [7]:
df_results['date'] = pd.to_datetime(df_results['date'])
df_elo['snapshot_date'] = pd.to_datetime(df_elo['snapshot_date'])

df_results = df_results.sort_values('date')
df_elo_sorted = df_elo.sort_values('snapshot_date')

##### Join

In [8]:
#Home team
df_matches = pd.merge_asof(
    df_results, df_elo_sorted[['country', 'snapshot_date', 'rating']].rename(columns={'country': 'elo_team'}),
    left_on='date', right_on='snapshot_date',
    left_by='home_team', right_by='elo_team',
    direction='backward'
).rename(columns={'rating': 'home_elo'}).drop(columns=['snapshot_date', 'elo_team'])

#Away team
df_matches = pd.merge_asof(
    df_matches, df_elo_sorted[['country', 'snapshot_date', 'rating']].rename(columns={'country': 'elo_team'}),
    left_on='date', right_on='snapshot_date',
    left_by='away_team', right_by='elo_team',
    direction='backward'
).rename(columns={'rating': 'away_elo'}).drop(columns=['snapshot_date', 'elo_team'])

For every match, look up each team's Elo rating **as it stood just before that match was played** — never a rating from after the fact, to avoid leaking the result into the feature.

`pd.merge_asof(..., direction='backward')` does this: for a given match date, it finds the most recent Elo snapshot on or before that date, matched within the same team (`left_by`/`right_by`). It's run twice — once for `home_team`, once for `away_team` — to produce `home_elo` and `away_elo`.

##### Final dataset

In [9]:
df_matches = df_matches[
    ['date', 'home_team', 'away_team', 'home_score', 'away_score',
     'tournament', 'neutral', 'home_elo', 'away_elo']
]

##### Removing matches where both elo's are not available

In [10]:
df_matches.isna().sum()

date              0
home_team         0
away_team         0
home_score        0
away_score        0
tournament        0
neutral           0
home_elo      32163
away_elo      33483
dtype: int64

In [11]:
df_matches = df_matches.dropna(subset=['home_elo', 'away_elo'])

In [12]:
df_matches.info()

<class 'pandas.DataFrame'>
Index: 7451 entries, 144 to 49390
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        7451 non-null   datetime64[us]
 1   home_team   7451 non-null   str           
 2   away_team   7451 non-null   str           
 3   home_score  7451 non-null   float64       
 4   away_score  7451 non-null   float64       
 5   tournament  7451 non-null   str           
 6   neutral     7451 non-null   bool          
 7   home_elo    7451 non-null   float64       
 8   away_elo    7451 non-null   float64       
dtypes: bool(1), datetime64[us](1), float64(4), str(3)
memory usage: 531.2 KB


`df_elo` only covers the **48 teams in the 2026 World Cup field** — not every federation that has ever played an international match. Since `df_results` contains 337 different teams across 150+ years, most historical matches involve at least one team outside that 48-team set, and so have no Elo rating available for one (or both) sides.

Checking the numbers: out of ~49,500 total matches, only **~7,500 (15%)** have a valid Elo rating for *both* teams. Requiring both ratings means dropping the other 85%.

This is treated as a deliberate scope decision, not a data-quality problem: this model will only ever be asked to predict matches between the 48 teams that could appear at the 2026 World Cup, so historical matches involving teams outside that set aren't relevant training signal for this task anyway.

##### Final dataset: `df_matches`

The result of merging `df_results` + `df_formernames` + `df_elo`, filtered down to matches where both teams have a valid pre-match Elo rating.

**7,451 matches**, 1908–2025, 9 columns:

| Column                 | Description                                            |
| ---------------------- | ------------------------------------------------------- |
| `date`                 | match date                                               |
| `home_team`, `away_team` | current team names (former names resolved)             |
| `home_score`, `away_score` | final scoreline — the prediction target              |
| `tournament`           | competition type (friendly, qualifier, World Cup, etc.) |
| `neutral`              | whether the match was played at a neutral venue         |
| `home_elo`, `away_elo` | each team's Elo rating as of just before the match      |

**Scope:** limited to the 48 teams that could appear at the 2026 World Cup — not the full 337-team historical record. This was a deliberate choice: those are the only teams this model will ever need to predict.

### Exploring `df_matches`

In [13]:
df_matches.info()

<class 'pandas.DataFrame'>
Index: 7451 entries, 144 to 49390
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        7451 non-null   datetime64[us]
 1   home_team   7451 non-null   str           
 2   away_team   7451 non-null   str           
 3   home_score  7451 non-null   float64       
 4   away_score  7451 non-null   float64       
 5   tournament  7451 non-null   str           
 6   neutral     7451 non-null   bool          
 7   home_elo    7451 non-null   float64       
 8   away_elo    7451 non-null   float64       
dtypes: bool(1), datetime64[us](1), float64(4), str(3)
memory usage: 531.2 KB


In [14]:
df_matches.describe()

,date,home_score,away_score,home_elo,away_elo
count,7451,7451.000000,7451.000000,7451.000000,7451.000000
mean,1988-01-22 16:00:42.517782,1.617367,1.177963,1765.955174,1760.146692
min,1902-05-03 00:00:00,0.000000,0.000000,973.000000,988.000000
25%,1970-06-08 12:00:00,1.000000,0.000000,1635.000000,1631.000000
50%,1995-09-20 00:00:00,1.000000,1.000000,1766.000000,1760.000000
75%,2010-07-03 00:00:00,2.000000,2.000000,1904.000000,1896.000000
max,2026-06-09 00:00:00,12.000000,11.000000,2213.000000,2207.000000
std,NaN,1.497206,1.252649,183.587709,179.741091
